# L7: Crie uma equipe para personalizar candidaturas a empregos¶
* Nesta lição, você criará seu primeiro sistema multiagente. 

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew

In [4]:
import yaml
import os 

with open("config.yaml", "r") as config_file:
  config = yaml.safe_load(config_file)
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]
os.environ["SERP_API_KEY"] = config["SERP_API_KEY"]
serp_api_key = os.getenv("SERP_API_KEY")

In [6]:
from crewai_tools import (
  FileReadTool,
  ScrapeWebsiteTool,
  MDXSearchTool,
  SerperDevTool
)

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
read_resume = FileReadTool(file_path='./fake_resume.md')
semantic_search_resume = MDXSearchTool(mdx='./fake_resume.md')

Inserting batches in chromadb: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]


In [7]:
from IPython.display import Markdown, display
display(Markdown("./fake_resume.md"))

# Ryan Johnson

[Email](mailto:ryan.johnson@example.com) | [LinkedIn](https://www.linkedin.com/in/ryanjohnson) | [GitHub](https://github.com/ryanjohnson)

---

## Resumo

Profissional dedicado e apaixonado por tecnologia com experiência em desenvolvimento de software, análise de dados e inteligência artificial. Possuo um forte histórico de entrega de soluções inovadoras, com foco em melhorar a eficiência e otimizar processos.

---

## Experiência Profissional

### **Desenvolvedor Full Stack**  
**Tech Solutions Ltda** | Janeiro de 2022 - Presente

- Liderança na criação e manutenção de aplicações web utilizando Python, Django, JavaScript e React.
- Desenvolvimento de APIs RESTful para integração de sistemas internos.
- Análise de requisitos e interação com clientes para garantir a entrega de soluções personalizadas.
- Colaboração em um time ágil, promovendo um ambiente de aprendizado contínuo e inovação.

### **Analista de Dados Júnior**  
**DataTech Corp** | Julho de 2020 - Dezembro de 2021

- Manipulação e análise de grandes volumes de dados usando Python e SQL.
- Criação de dashboards e relatórios para tomada de decisão usando ferramentas como Power BI e Tableau.
- Desenvolvimento de modelos preditivos e testes A/B para otimização de processos empresariais.

---

## Educação

**Bacharel em Ciência da Computação**  
**Universidade XYZ** | Conclusão em 2020

---

## Habilidades Técnicas

- **Linguagens de Programação**: Python, JavaScript, SQL, HTML/CSS
- **Frameworks**: Django, React, Flask
- **Ferramentas de Análise de Dados**: Pandas, NumPy, Matplotlib, Power BI, Tableau
- **Bases de Dados**: MySQL, PostgreSQL, MongoDB
- **Outras Tecnologias**: Git, Docker, AWS, Kubernetes

---

## Projetos

### **Sistema de Recomendação de Filmes**
- Desenvolvimento de um sistema de recomendação baseado em aprendizado de máquina utilizando Python e scikit-learn.
- Implementação de técnicas de filtragem colaborativa e baseada em conteúdo.

### **Aplicativo de Tarefas**
- Criação de um aplicativo de gerenciamento de tarefas com front-end em React e back-end em Django.
- Implementação de funcionalidades como autenticação de usuários e armazenamento de dados em PostgreSQL.

---

## Certificações

- **Certificado em Machine Learning** - Coursera, 2021
- **Certificado em Desenvolvimento Web Full Stack** - Udemy, 2020

---

## Idiomas

- **Inglês**: Fluente
- **Espanhol**: Intermediário

---

## Interesses

- Inteligência Artificial
- Desenvolvimento Web
- Análise de Dados


# Creating Agents

In [8]:
# Agente 1: Pesquisador
researcher = Agent(
    role="Pesquisador de Vagas de Emprego",
    goal="Realizar uma análise incrível das "
         "publicações de vagas para ajudar os candidatos a emprego",
    tools=[scrape_tool, search_tool],
    verbose=True,
    backstory=(
        "Como Pesquisador de Vagas, sua habilidade em "
        "navegar e extrair informações críticas "
        "de publicações de vagas é incomparável. "
        "Suas habilidades ajudam a identificar as qualificações "
        "e competências necessárias exigidas pelos empregadores, "
        "formando a base para uma personalização eficaz das candidaturas."
    )
)

In [9]:
# Agente 2: Perfilador
profiler = Agent(
    role="Perfilador Pessoal para Engenheiros",
    goal="Realizar uma pesquisa incrível sobre candidatos a emprego "
         "para ajudá-los a se destacar no mercado de trabalho",
    tools=[scrape_tool, search_tool,
           read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Equipado com grande poder analítico, você disseca "
        "e sintetiza informações de diversas fontes para criar "
        "perfis pessoais e profissionais abrangentes, "
        "preparando o terreno para aprimoramentos personalizados de currículos."
    )
)

In [10]:
# Agente 3: Estrategista de Currículos
resume_strategist = Agent(
    role="Estrategista de Currículos para Engenheiros",
    goal="Encontrar todas as melhores maneiras de fazer um "
         "currículo se destacar no mercado de trabalho.",
    tools=[scrape_tool, search_tool,
           read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Com uma mente estratégica e um olhar atento aos detalhes, você "
        "se destaca em refinar currículos para destacar as habilidades "
        "e experiências mais relevantes, garantindo que eles "
        "se alinhem perfeitamente com os requisitos da vaga."
    )
)

In [11]:
# Agente 4: Preparador de Entrevistas
interview_preparer = Agent(
    role="Preparador de Entrevistas para Engenharia",
    goal="Criar perguntas de entrevista e pontos de discussão "
         "com base no currículo e nos requisitos da vaga",
    tools=[scrape_tool, search_tool,
           read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Seu papel é crucial para antecipar a dinâmica das "
        "entrevistas. Com sua habilidade em formular perguntas-chave "
        "e pontos de discussão, você prepara os candidatos para o sucesso, "
        "garantindo que eles possam abordar com confiança todos os aspectos da "
        "vaga para a qual estão se candidatando."
    )
)

# Creating Tasks

In [12]:
# Tarefa para o Agente Pesquisador: Extrair Requisitos de Vaga
research_task = Task(
    description=(
        "Analise a URL da vaga fornecida ({job_posting_url}) "
        "para extrair as principais habilidades, experiências e qualificações "
        "necessárias. Utilize as ferramentas para coletar conteúdo e identificar "
        "e categorizar os requisitos."
    ),
    expected_output=(
        "Uma lista estruturada de requisitos da vaga, incluindo habilidades, "
        "qualificações e experiências necessárias."
    ),
    agent=researcher,
    async_execution=True
)

In [13]:
# Tarefa para o Agente Perfilador: Compilar Perfil Abrangente
profile_task = Task(
    description=(
        "Compile um perfil detalhado pessoal e profissional "
        "usando as URLs do GitHub ({github_url}) e a descrição pessoal "
        "({personal_writeup}). Utilize as ferramentas para extrair e "
        "sintetizar informações dessas fontes."
    ),
    expected_output=(
        "Um documento de perfil abrangente que inclui habilidades, "
        "experiências em projetos, contribuições, interesses e "
        "estilo de comunicação."
    ),
    agent=profiler,
    async_execution=True
)

* Você pode passar uma lista de tarefas como contexto para uma tarefa.
* A tarefa então leva em conta a saída dessas tarefas em sua execução.
* A tarefa não será executada até que tenha a(s) saída(s) dessas tarefas.

In [14]:
# Tarefa para o Agente Estratégia de Currículo: Alinhar Currículo com Requisitos da Vaga
resume_strategy_task = Task(
    description=(
        "Usando o perfil e os requisitos da vaga obtidos nas "
        "tarefas anteriores, personalize o currículo para destacar as áreas "
        "mais relevantes. Utilize as ferramentas para ajustar e aprimorar o "
        "conteúdo do currículo. Certifique-se de que este seja o melhor currículo, "
        "mas não invente nenhuma informação. Atualize todas as seções, "
        "incluindo o resumo inicial, experiência de trabalho, habilidades, "
        "e educação. Tudo para refletir melhor as habilidades do candidato "
        "e como elas correspondem à vaga."
    ),
    expected_output=(
        "Um currículo atualizado que destaca eficazmente as qualificações "
        "e experiências do candidato relevantes para a vaga."
    ),
    output_file="curriculo_personalizado.md",
    context=[research_task, profile_task],
    agent=resume_strategist
)

In [15]:
# Tarefa para o Agente Preparador de Entrevistas: Desenvolver Materiais para a Entrevista
interview_preparation_task = Task(
    description=(
        "Crie um conjunto de perguntas potenciais para a entrevista e pontos "
        "de discussão com base no currículo personalizado e nos requisitos da vaga. "
        "Utilize as ferramentas para gerar perguntas e pontos relevantes. "
        "Certifique-se de usar essas perguntas e pontos de discussão para "
        "ajudar o candidato a destacar os principais pontos do currículo "
        "e como ele se alinha com a vaga."
    ),
    expected_output=(
        "Um documento contendo as perguntas chave e pontos de discussão "
        "que o candidato deve preparar para a entrevista inicial."
    ),
    output_file="materiais_entrevista.md",
    context=[research_task, profile_task, resume_strategy_task],
    agent=interview_preparer
)

# Creating the Crew

In [16]:
job_application_crew = Crew(
    agents=[researcher,
            profiler,
            resume_strategist,
            interview_preparer],

    tasks=[research_task,
           profile_task,
           resume_strategy_task,
           interview_preparation_task],

    verbose=True
)

# Running the Crew

In [18]:
job_application_inputs = {
    'job_posting_url': 'https://jobs.lever.co/AIFund/6c82e23e-d954-4dd8-a734-c0c2c5ee00f1?lever-origin=applied&lever-source%5B%5D=AI+Fund',
    'github_url': 'https://github.com/joaomdmoura',
    'personal_writeup': """Ryan é um Líder em Engenharia de Software
    com 18 anos de experiência, especializado em
    gerenciar equipes remotas e presenciais, além de ser um expert em múltiplas
    linguagens de programação e frameworks. Ele possui um MBA e uma sólida
    formação em IA e ciência de dados. Noah liderou com sucesso
    grandes iniciativas tecnológicas e startups, provando sua habilidade de impulsionar
    inovação e crescimento na indústria de tecnologia. Ideal para funções de liderança
    que exigem uma abordagem estratégica e inovadora."""
}

In [19]:
### this execution will take a few minutes to run
result = job_application_crew.kickoff(inputs=job_application_inputs)

 [DEBUG]: == Working Agent: Pesquisador de Vagas de Emprego
 [INFO]: == Starting Task: Analise a URL da vaga fornecida (https://jobs.lever.co/AIFund/6c82e23e-d954-4dd8-a734-c0c2c5ee00f1?lever-origin=applied&lever-source%5B%5D=AI+Fund) para extrair as principais habilidades, experiências e qualificações necessárias. Utilize as ferramentas para coletar conteúdo e identificar e categorizar os requisitos.
 [DEBUG]: == [Pesquisador de Vagas de Emprego] Task output: 


 [DEBUG]: == Working Agent: Perfilador Pessoal para Engenheiros
 [INFO]: == Starting Task: Compile um perfil detalhado pessoal e profissional usando as URLs do GitHub (https://github.com/joaomdmoura) e a descrição pessoal (Ryan é um Líder em Engenharia de Software
    com 18 anos de experiência, especializado em
    gerenciar equipes remotas e presenciais, além de ser um expert em múltiplas
    linguagens de programação e frameworks. Ele possui um MBA e uma sólida
    formação em IA e ciência de dados. Noah liderou com sucesso

In [ ]:
from IPython.display import Markdown, display
display(Markdown("./tailored_resume.md"))

In [ ]:
display(Markdown("./interview_materials.md"))